# Task 1: Domain-Holdout LoRA Fine-Tuning & 7B Memory Analysis
**Applied Generative AI Engineering & System Optimization**
*Target Model: Qwen/Qwen2.5-0.5B-Instruct | Task: Abstract -> Structured JSON Extraction*

This notebook executes:
1. Domain-holdout dataset generation (8 train domains, 4 unseen test domains) with zero-leakage verification.
2. Base model zero-shot evaluation on the held-out test set.
3. LoRA fine-tuning ($r=16, \alpha=32$, all linear modules adapted, cosine schedule).
4. Checkpoint saving and held-out evaluation.
5. Stress testing on 3 difficult edge-case failure modes.
6. Rigorous 7B GPU memory requirement calculations (Full FT, LoRA, QLoRA).


In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q torch transformers peft datasets accelerate bitsandbytes

import os
import json
import time
import random
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


## 2. Domain-Holdout Data Generation (Zero-Leakage Guarantee)
- **Train + Val (8 domains)**: Machine Translation, Language Modeling, Image Classification, Object Detection, Semantic Segmentation, Question Answering, Named Entity Recognition, Sentiment Analysis.
- **Held-Out Test (4 completely unseen domains)**: Speech Recognition, Code Generation, Reinforcement Learning, Protein Structure Prediction.
- **Distractor Injection**: Parameter counts (e.g. `65M params`), throughput speedups (`2.4x`), and performance margins (`3.4%`) to evaluate true numerical grounding.


In [ ]:
# Execute data generation with domain-holdout
!python ../task1_finetune/prepare_data_fixed.py


## 3. Evaluation Metrics & Base Model Zero-Shot Baseline
We evaluate field-by-field token-level Precision, Recall, $F_1$, and Overall Macro $F_1$ on the 50 held-out unseen-domain test samples.


In [ ]:
# Evaluate Base Model Zero-Shot Baseline
!python ../task1_finetune/eval_base.py


## 4. LoRA Fine-Tuning Execution
- **Rank ($r = 16$) & Alpha ($\alpha = 32$)**: Stable scaling multiplier $\alpha/r = 2.0$.
- **Target Modules**: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` (Full attention + MLP projections for format-locking JSON).
- **Optimization**: AdamW, Cosine LR scheduler, initial $\text{LR} = 2 \times 10^{-4}$, 4 epochs.


In [ ]:
# Train LoRA Adapter & Save Checkpoints
!python ../task1_finetune/train_lora.py


## 5. Fine-Tuned LoRA Evaluation on Held-Out Test Set
Evaluate the adapter on the 4 unseen domains and compute the empirical delta over the base model.


In [ ]:
# Evaluate Fine-Tuned Model
!python ../task1_finetune/eval_lora.py


## 6. Stress Testing & 3 Failure Modes Analysis
Evaluate on edge cases:
1. Multi-metric & multi-dataset competition (relational coupling loss).
2. Purely qualitative claims without numbers (hallucination under ungrounded input).
3. Relative gain vs absolute score conflation.


In [ ]:
# Run failure mode stress tests
!python ../task1_finetune/stress_test_failures.py


## 7. 7B Model GPU Memory Derivation (All Working Shown)
*Setup: 7.0 Billion parameter model ($N = 7.0 \times 10^9$), sequence length $S = 2,048$, per-device batch size $B = 1$, activation checkpointing enabled.*

1. **(a) Full Fine-Tuning (16-bit Mixed Precision AdamW)**:
   - Weights (FP16): $7.0 \times 10^9 \times 2\text{ bytes} = 14.00\text{ GB}$
   - Gradients (FP16): $7.0 \times 10^9 \times 2\text{ bytes} = 14.00\text{ GB}$
   - Optimizer States (AdamW FP32): Momentum $m$ + Variance $v$ ($8\text{ bytes/param}$) $= 56.00\text{ GB}$
   - Checkpointed Activations ($B=1, S=2048$): $\approx 5.00\text{ GB}$
   - **Total Required VRAM: 89.00 GB** $\to$ Requires $2\times$ A100-80GB or $1\times$ 80GB with ZeRO-3 offload.

2. **(b) LoRA Fine-Tuning ($r = 16$, 16-bit Base Weights)**:
   - Base Weights (FP16, Frozen): $14.00\text{ GB}$ (no base gradients or optimizer states)
   - Trainable LoRA Params (0.28%): $19.6 \times 10^6$ parameters
   - LoRA Weights + Gradients: $19.6 \times 10^6 \times 4\text{ bytes} \approx 0.08\text{ GB}$
   - LoRA Optimizer States (FP32 AdamW): $19.6 \times 10^6 \times 8\text{ bytes} \approx 0.16\text{ GB}$
   - Activations + Dynamic Overhead: $\approx 5.37\text{ GB}$
   - **Total Required VRAM: 19.61 GB** $\to$ Fits comfortably on a single 24 GB RTX 3090/4090.

3. **(c) QLoRA Fine-Tuning (4-bit NF4 Base Weights, Double Quantization, Paged AdamW)**:
   - Base Weights (4-bit NF4): $7.0 \times 10^9 \times 0.5\text{ bytes} = 3.50\text{ GB}$
   - Double Quantization & Scaling Constants: $\approx 0.45\text{ GB}$
   - LoRA Parameters, Gradients & Paged Optimizer: $\approx 0.48\text{ GB}$ (paged into CPU RAM during memory spikes)
   - Checkpointed Activations & CUDA Buffers: $\approx 4.91\text{ GB}$
   - **Total Required VRAM: 9.34 GB** $\to$ Fits easily on consumer 12 GB/16 GB GPUs (RTX 4070/4080).
